# Ultimate Alpha v16 — Dryrun Diagnostic

This notebook reproduces the analysis that identified the v16_risk_on failure mode
and drove the June 2026 regime gate change.

**Data source:** Freqtrade SQLite dryrun database (139 closed trades, April–May 2026)

**Finding:** The `risk_on` regime — intended to be the most favorable entry condition —
had a 23% win rate across 116 trades. The `neutral` regime had a 75% win rate.
The gate was updated accordingly.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Update this path to point to your local dryrun database
DB_PATH = '../../../vibe_os/user_data/trades_dryrun.sqlite'

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    'SELECT * FROM trades WHERE close_date IS NOT NULL',
    conn
)
print(f'Loaded {len(df)} closed trades')
print(f'Period: {df.open_date.min()[:10]} to {df.close_date.max()[:10]}')
print(f'Pairs: {df.pair.nunique()} unique')

## 1. Overall performance

In [ ]:
total = len(df)
wins = (df.close_profit > 0).sum()
pnl = df.close_profit_abs.sum()

print(f'Total trades:  {total}')
print(f'Win rate:      {wins/total*100:.1f}% ({wins}/{total})')
print(f'Net PnL (USDT): {pnl:.2f}')
print(f'Avg win:       +{df[df.close_profit>0].close_profit.mean()*100:.2f}%')
print(f'Avg loss:       {df[df.close_profit<=0].close_profit.mean()*100:.2f}%')

## 2. Performance by regime at entry

This is the key finding. The `enter_tag` column records which regime
the strategy was in when it opened each trade.

In [ ]:
regime_stats = df.groupby('enter_tag').agg(
    trades=('close_profit', 'count'),
    wins=('close_profit', lambda x: (x > 0).sum()),
    net_pnl=('close_profit_abs', 'sum'),
    avg_pct=('close_profit', 'mean')
).reset_index()

regime_stats['win_rate'] = regime_stats['wins'] / regime_stats['trades'] * 100
regime_stats['avg_pct'] = regime_stats['avg_pct'] * 100

print(regime_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Performance by Regime at Entry', fontsize=14, fontweight='bold')

colors = ['#e74c3c' if w < 40 else '#2ecc71' for w in regime_stats['win_rate']]

# Win rate chart
ax1 = axes[0]
bars = ax1.bar(regime_stats['enter_tag'], regime_stats['win_rate'], color=colors, edgecolor='white', linewidth=0.5)
ax1.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Break-even (~50%)')
ax1.set_ylabel('Win Rate (%)')
ax1.set_title('Win Rate by Regime')
ax1.set_ylim(0, 100)
for bar, val in zip(bars, regime_stats['win_rate']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontweight='bold')
ax1.legend()

# Net PnL chart
ax2 = axes[1]
pnl_colors = ['#e74c3c' if p < 0 else '#2ecc71' for p in regime_stats['net_pnl']]
bars2 = ax2.bar(regime_stats['enter_tag'], regime_stats['net_pnl'], color=pnl_colors, edgecolor='white', linewidth=0.5)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.8)
ax2.set_ylabel('Net PnL (USDT)')
ax2.set_title('Net PnL by Regime')
for bar, val in zip(bars2, regime_stats['net_pnl']):
    ypos = bar.get_height() + 0.5 if val >= 0 else bar.get_height() - 2
    ax2.text(bar.get_x() + bar.get_width()/2, ypos,
             f'${val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('notebooks/regime_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to notebooks/regime_performance.png')

## 3. Exit reason breakdown

In [ ]:
exit_stats = df.groupby('exit_reason').agg(
    trades=('close_profit', 'count'),
    wins=('close_profit', lambda x: (x > 0).sum()),
    net_pnl=('close_profit_abs', 'sum')
).reset_index()
exit_stats['win_rate'] = exit_stats['wins'] / exit_stats['trades'] * 100
print(exit_stats.to_string(index=False))

print()
print('Key insight: ROI exits are 100% wins. Stoploss exits are 11% wins.')
print('Problem is not exit logic — it is entry quality in risk_on regime.')

## 4. The fix applied

Based on this analysis, the regime gate was updated on June 6, 2026:

```python
# Before: risk_on was the most permissive regime
regime_ok = (regime != 'risk_off') | (df['final_score'] > 0.75)

# After: mirror-symmetric — only neutral trades freely
if regime == 'neutral':
    regime_ok = pd.Series(True, index=df.index)
else:  # risk_on OR risk_off
    regime_ok = df['final_score'] > 0.75
```

**Rationale:** The data showed neutral (75% win rate) and risk_on (23% win rate).
The strategy was most permissive where it should have been most restrictive.
The fix requires exceptional score in both risk_on and risk_off, and allows
free entry only in the one regime that demonstrated positive expected value.

This is the correct engineering approach: change one thing, driven by data,
with a clear hypothesis, then accumulate more data to verify.

## 5. Current status

The bot is running under the updated gate. Since the change, the regime
classifier has been in `risk_off` — the strategy is correctly refusing to
enter trades because current market conditions don't meet the new bar.

Next meaningful review: when 50+ trades have accumulated under the new gate.